In [1]:
# %%
import pandas as pd
import numpy as np
import glob
import os
from IPython.display import display

print("🚀 Step 1: Loading Raw Data...")

DATA_RAW = os.path.join("..", "data", "raw")
DATA_PROCESSED = os.path.join("..", "data", "processed")

# Ensure processed directory exists
os.makedirs(DATA_PROCESSED, exist_ok=True)

# Select exactly the columns we need to build our 10-30 feature set
# Household features: PUMA, SNAP, Internet, Vehicles, Language, Rent Burden, Rooms, Tenure, HH Income, HH Size
h_cols = ['SERIALNO', 'PUMA', 'FS', 'FFSP', 'ACCESSINET', 'VEH', 'LNGI', 'GRPIP', 'RMSP', 'TEN', 'HINCP', 'NP']
# Person features: Poverty Ratio, Age, Disability, Education, Employment, Race, Ethnicity
p_cols = ['SERIALNO', 'POVPIP', 'AGEP', 'DIS', 'SCHL', 'ESR', 'RAC1P', 'HISP']

h_files = glob.glob(os.path.join(DATA_RAW, "psam_h*.csv"))
p_files = glob.glob(os.path.join(DATA_RAW, "psam_p*.csv"))

if not h_files or not p_files:
    raise FileNotFoundError("Missing PUMS files in data/raw. Please run the download script first.")

df_h_raw = pd.concat([pd.read_csv(f, usecols=h_cols) for f in h_files], ignore_index=True)
df_p_raw = pd.concat([pd.read_csv(f, usecols=p_cols) for f in p_files], ignore_index=True)

print(f"✅ Loaded Households: {df_h_raw.shape}")
print(f"✅ Loaded People: {df_p_raw.shape}")

🚀 Step 1: Loading Raw Data...
✅ Loaded Households: (74024, 12)
✅ Loaded People: (157694, 8)


In [2]:
# %%
print("⚙️ Step 2: Engineering Person-Level Barrier Features...")

df_p = df_p_raw.copy()

# 1. Create binary flags for individuals
df_p['is_elderly'] = (df_p['AGEP'] >= 60).astype(int)
df_p['is_child'] = (df_p['AGEP'] < 18).astype(int)
df_p['is_disabled'] = (df_p['DIS'] == 1).astype(int) # 1 = Yes, 2 = No
df_p['is_working'] = df_p['ESR'].isin([1, 2, 4, 5]).astype(int) # Employed codes
df_p['is_minority'] = ((df_p['RAC1P'] != 1) | (df_p['HISP'] != 1)).astype(int) # Not strictly non-Hispanic White
df_p['SCHL'] = df_p['SCHL'].fillna(0) # Fill NaN education with 0

# 2. Aggregate to Household Level (SERIALNO)
hh_agg = df_p.groupby('SERIALNO').agg(
    POVPIP=('POVPIP', 'first'),              # Use first person's poverty ratio for the household
    HAS_ELDERLY=('is_elderly', 'max'),       # 1 if ANYONE is elderly
    HAS_DISABLED=('is_disabled', 'max'),     # 1 if ANYONE is disabled
    NUM_CHILDREN=('is_child', 'sum'),        # Count of children
    NUM_WORKING_ADULTS=('is_working', 'sum'),# Count of working adults
    MAX_EDUCATION=('SCHL', 'max'),           # Highest education level in the house
    IS_MINORITY_HH=('is_minority', 'max')    # 1 if anyone identifies as a minority
).reset_index()

print("✅ Aggregated person demographics to household level.")

⚙️ Step 2: Engineering Person-Level Barrier Features...
✅ Aggregated person demographics to household level.


In [3]:
# %%
print("⚙️ Step 3: Merging & Filtering for Low-Income Households...")

# 1. Merge the aggregated person data into the household data
df_merged = pd.merge(df_h_raw, hh_agg, on='SERIALNO', how='inner')

# 2. Apply Filters:
# - Remove Group Quarters (GQ)
# - Remove imputed SNAP values (FFSP == 1)
# - STRICTLY LOW INCOME: POVPIP <= 130
mask_standard = ~df_merged['SERIALNO'].astype(str).str.contains('GQ')
mask_real_snap = df_merged['FFSP'] != 1
mask_low_income = df_merged['POVPIP'] <= 130

df_clean = df_merged[mask_standard & mask_real_snap & mask_low_income].copy()

# 3. Create the Target Variable `y` (1 = Receives SNAP, 0 = Does NOT receive SNAP)
# FS == 1 means Yes, FS == 2 means No.
df_clean['y'] = df_clean['FS'].apply(lambda x: 1 if x == 1.0 else 0)

# 4. Handle Missing Values / Formatting for Machine Learning
df_clean['VEH'] = df_clean['VEH'].fillna(0)  # Assume 0 vehicles if NaN
df_clean['GRPIP'] = df_clean['GRPIP'].fillna(0) # 0 rent burden if NaN (e.g., owns outright without mortgage)
df_clean['ACCESSINET'] = df_clean['ACCESSINET'].fillna(3) # 3 = No internet access
df_clean['LNGI'] = df_clean['LNGI'].fillna(1) # 1 = Not limited, 2 = Limited
df_clean['TEN'] = df_clean['TEN'].fillna(0) # Tenure/Ownership

# 5. Final Feature Selection
final_columns = [
    'SERIALNO', 'PUMA', 'y', 'POVPIP', 'HAS_ELDERLY', 'HAS_DISABLED', 
    'NUM_CHILDREN', 'NUM_WORKING_ADULTS', 'MAX_EDUCATION', 'IS_MINORITY_HH',
    'VEH', 'ACCESSINET', 'LNGI', 'GRPIP', 'RMSP', 'TEN', 'HINCP', 'NP'
]

df_final = df_clean[final_columns].copy()

print(f"✅ Filtered to strictly eligible standard households: {df_final.shape[0]} rows.")

⚙️ Step 3: Merging & Filtering for Low-Income Households...
✅ Filtered to strictly eligible standard households: 7450 rows.


In [4]:
# %%
print("📝 Step 4: Generating Data Dictionary...")

# Define what each column means and its type
doc_data = [
    {"Column": "SERIALNO", "Description": "Unique Household Identifier", "Type": "String"},
    {"Column": "PUMA", "Description": "Public Use Microdata Area (Geography Code)", "Type": "Categorical/Numeric"},
    {"Column": "y", "Description": "TARGET: 1 if receives SNAP, 0 if in the GAP (Eligible but no SNAP)", "Type": "Binary (0/1)"},
    {"Column": "POVPIP", "Description": "Income-to-Poverty Ratio (0 to 130)", "Type": "Numeric"},
    {"Column": "HAS_ELDERLY", "Description": "1 if household has someone 60+", "Type": "Binary (0/1)"},
    {"Column": "HAS_DISABLED", "Description": "1 if household has someone with a disability", "Type": "Binary (0/1)"},
    {"Column": "NUM_CHILDREN", "Description": "Number of people under 18 in the household", "Type": "Numeric"},
    {"Column": "NUM_WORKING_ADULTS", "Description": "Number of employed adults in the household", "Type": "Numeric"},
    {"Column": "MAX_EDUCATION", "Description": "Highest educational attainment code in the household (1-24)", "Type": "Numeric (Ordinal)"},
    {"Column": "IS_MINORITY_HH", "Description": "1 if anyone in the household identifies as a racial/ethnic minority", "Type": "Binary (0/1)"},
    {"Column": "VEH", "Description": "Number of vehicles available (0-6)", "Type": "Numeric"},
    {"Column": "ACCESSINET", "Description": "Internet access: 1/2=Yes, 3=No", "Type": "Categorical"},
    {"Column": "LNGI", "Description": "Limited English speaking household: 1=Not limited, 2=Limited", "Type": "Categorical"},
    {"Column": "GRPIP", "Description": "Gross rent as a percentage of household income", "Type": "Numeric"},
    {"Column": "RMSP", "Description": "Number of rooms in the housing unit", "Type": "Numeric"},
    {"Column": "TEN", "Description": "Housing tenure (1/2=Own, 3/4=Rent)", "Type": "Categorical"},
    {"Column": "HINCP", "Description": "Household income in past 12 months", "Type": "Numeric"},
    {"Column": "NP", "Description": "Number of person records in the household", "Type": "Numeric"}
]

df_documentation = pd.DataFrame(doc_data)
display(df_documentation)

📝 Step 4: Generating Data Dictionary...


,Column,Description,Type
0,SERIALNO,Unique Household Identifier,String
1,PUMA,Public Use Microdata Area (Geography Code),Categorical/Numeric
2,y,"TARGET: 1 if receives SNAP, 0 if in the GAP (E...",Binary (0/1)
3,POVPIP,Income-to-Poverty Ratio (0 to 130),Numeric
4,HAS_ELDERLY,1 if household has someone 60+,Binary (0/1)
5,HAS_DISABLED,1 if household has someone with a disability,Binary (0/1)
6,NUM_CHILDREN,Number of people under 18 in the household,Numeric
7,NUM_WORKING_ADULTS,Number of employed adults in the household,Numeric
8,MAX_EDUCATION,Highest educational attainment code in the hou...,Numeric (Ordinal)
9,IS_MINORITY_HH,1 if anyone in the household identifies as a r...,Binary (0/1)


In [5]:
# %%
print("💾 Step 5: Sampling, Displaying, and Saving Results...")

# 1. Split into the GAP and Claiming households
df_gap = df_final[df_final['y'] == 0]      # Eligible but NOT claiming
df_claiming = df_final[df_final['y'] == 1] # Eligible AND claiming

# 2. Select 20 random rows from each
sample_gap = df_gap.sample(n=min(20, len(df_gap)), random_state=42)
sample_claiming = df_claiming.sample(n=min(20, len(df_claiming)), random_state=42)

print("\n🔴 THE GAP HOUSEHOLDS (y = 0) - Sample of 20:")
display(sample_gap)

print("\n🔵 CLAIMING HOUSEHOLDS (y = 1) - Sample of 20:")
display(sample_claiming)

# 3. Concatenate and save to processed data folder
df_processed_out = pd.concat([df_gap, df_claiming], ignore_index=True)

out_path = os.path.join(DATA_PROCESSED, "processed_low_income_households.csv")
df_processed_out.to_csv(out_path, index=False)

# Also save the documentation for easy reference
doc_path = os.path.join(DATA_PROCESSED, "data_dictionary.csv")
df_documentation.to_csv(doc_path, index=False)

print(f"\n✅ SUCCESS: Processed dataset saved to {out_path} ({df_processed_out.shape[0]} rows)")
print(f"✅ SUCCESS: Data Dictionary saved to {doc_path}")

💾 Step 5: Sampling, Displaying, and Saving Results...

🔴 THE GAP HOUSEHOLDS (y = 0) - Sample of 20:


,SERIALNO,PUMA,y,POVPIP,HAS_ELDERLY,HAS_DISABLED,NUM_CHILDREN,NUM_WORKING_ADULTS,MAX_EDUCATION,IS_MINORITY_HH,VEH,ACCESSINET,LNGI,GRPIP,RMSP,TEN,HINCP,NP
68858,2023HU1389337,7300,0,128.0,1,1,0,1,11.0,1,2.0,3.0,1.0,0.0,4.0,2.0,20000.0,1
52089,2023HU0699551,71002,0,113.0,1,0,0,0,21.0,0,0.0,1.0,1.0,53.0,3.0,3.0,16300.0,1
15488,2023HU0551209,601,0,0.0,0,0,0,0,20.0,0,2.0,1.0,1.0,0.0,5.0,1.0,0.0,1
61885,2023HU1100499,70000,0,124.0,0,1,0,0,19.0,0,1.0,1.0,1.0,0.0,8.0,1.0,25000.0,2
53989,2023HU0775965,8702,0,0.0,1,0,0,0,24.0,0,2.0,1.0,1.0,0.0,9.0,2.0,0.0,2
66934,2023HU1308924,17100,0,102.0,0,0,0,1,16.0,0,2.0,1.0,1.0,0.0,4.0,1.0,16000.0,1
46314,2023HU0453626,16500,0,5.0,0,1,0,0,20.0,0,0.0,2.0,1.0,101.0,1.0,3.0,900.0,1
20880,2023HU0869987,1107,0,6.0,1,0,0,0,13.0,0,1.0,1.0,1.0,0.0,7.0,4.0,1000.0,1
64496,2023HU1206102,71001,0,125.0,0,0,1,1,19.0,0,1.0,1.0,1.0,0.0,9.0,1.0,26000.0,2
1846,2023HU0568474,103,0,93.0,1,1,0,2,21.0,1,1.0,1.0,1.0,66.0,5.0,3.0,29100.0,4



🔵 CLAIMING HOUSEHOLDS (y = 1) - Sample of 20:


,SERIALNO,PUMA,y,POVPIP,HAS_ELDERLY,HAS_DISABLED,NUM_CHILDREN,NUM_WORKING_ADULTS,MAX_EDUCATION,IS_MINORITY_HH,VEH,ACCESSINET,LNGI,GRPIP,RMSP,TEN,HINCP,NP
7418,2023HU0063613,507,1,37.0,1,0,0,0,21.0,1,1.0,3.0,2.0,48.0,1.0,3.0,5300.0,1
28455,2023HU1320904,1107,1,19.0,1,1,0,1,19.0,1,0.0,1.0,1.0,101.0,2.0,3.0,3770.0,2
35470,2023HU0005660,7300,1,0.0,1,1,0,0,11.0,1,0.0,1.0,1.0,0.0,8.0,2.0,0.0,1
12437,2023HU0363100,805,1,97.0,0,0,3,1,9.0,1,1.0,2.0,2.0,49.0,6.0,3.0,34300.0,5
63109,2023HU1150370,14300,1,121.0,1,1,0,0,11.0,0,1.0,3.0,1.0,0.0,6.0,2.0,17500.0,1
6456,2023HU0005676,400,1,62.0,0,1,0,1,21.0,0,1.0,1.0,1.0,0.0,5.0,1.0,9900.0,1
41715,2023HU0262281,71002,1,70.0,1,1,0,0,19.0,1,0.0,1.0,1.0,32.0,6.0,3.0,22000.0,4
69227,2023HU1404378,70000,1,28.0,0,1,0,0,9.0,1,1.0,1.0,1.0,0.0,13.0,1.0,4500.0,1
20468,2023HU0847340,802,1,64.0,1,1,1,0,15.0,1,2.0,1.0,1.0,0.0,6.0,2.0,15600.0,3
59410,2023HU0998218,5901,1,130.0,0,0,0,1,17.0,1,1.0,1.0,1.0,24.0,4.0,3.0,20000.0,1



✅ SUCCESS: Processed dataset saved to ../data/processed/processed_low_income_households.csv (7450 rows)
✅ SUCCESS: Data Dictionary saved to ../data/processed/data_dictionary.csv
